In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding,
    pipeline
)
from sklearn.utils.class_weight import compute_class_weight

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1. VERİ YÜKLEME VE ÖN İŞLEME
print("Veri yükleniyor ve ön işleme yapılıyor...")
df = pd.read_csv("data.csv", low_memory=False)



# Nutriscore harflere çevir
if df["nutriscore_notu"].dtype in [float, int]:
    df["nutriscore_notu"] = df["nutriscore_notu"].map(
        {0.0: "A", 1.0: "B", 2.0: "C", 3.0: "D", 4.0: "E"}
    )

mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
df["label"] = df["nutriscore_notu"].map(mapping)
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

df["text"] = (
    "Energy: " + df["enerji_kcal"].fillna(0).astype(str) + " kcal. "
    + "Fat: " + df["yag_g"].fillna(0).astype(str) + "g. "
    + "Saturated fat: " + df["doymus_yag_g"].fillna(0).astype(str) + "g. "
    + "Carbs: " + df["karbonhidrat_g"].fillna(0).astype(str) + "g. "
    + "Sugar: " + df["seker_g"].fillna(0).astype(str) + "g. "
    + "Fiber: " + df["lif_g"].fillna(0).astype(str) + "g. "
    + "Protein: " + df["protein_g"].fillna(0).astype(str) + "g. "
    + "Salt: " + df["tuz_g"].fillna(0).astype(str) + "g. "
    + "Nova group: " + df["nova_grubu"].fillna(0).astype(str) + "."
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2, 3, 4]),
    y=df["label"].values
)
print("Sınıf Ağırlıkları:", dict(zip(["A","B","C","D","E"], class_weights.round(2))))
print("Veri boyutu:", df.shape)

Veri yükleniyor ve ön işleme yapılıyor...


In [ ]:
class NutriDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


class WeightedTrainer(Trainer):
    def __init__(self, class_weights_tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_tensor = class_weights_tensor

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(
            weight=self.class_weights_tensor.to(logits.device)
        )
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

print("Sınıflar tanımlandı.")

In [ ]:
# Eğitim, Doğrulama ve Test setlerine ayırma (%80 Train, %10 Val, %10 Test)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

In [ ]:
# Pandas DataFrame'leri HuggingFace Dataset formatına çeviriyoruz
hg_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'validation': Dataset.from_pandas(val_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# 2. TOKENİZASYON (DeBERTa v3 Base)
print("Tokenizasyon işlemi başlatılıyor...")
model_name = "microsoft/deberta-v3-base" 
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized_datasets = hg_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# 3. METRİKLERİN TANIMLANMASI (Evaluate yerine tamamen Sklearn kullanıyoruz)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    
    return {"accuracy": acc, "f1": f1}

In [ ]:
# 4. MODEL YÜKLEME
print("Model yükleniyor...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels,
    id2label={i: label for i, label in enumerate(label_encoder.classes_)},
    label2id={label: i for i, label in enumerate(label_encoder.classes_)}
)

In [ ]:
# 5. EĞİTİM ARGÜMANLARI (DeBERTa FP16 Hatasından Arındırıldı)
training_args = TrainingArguments(
    output_dir="./deberta-nutriscore-model",
    learning_rate=2e-5,                  
    per_device_train_batch_size=8,       # Hafıza taşmasın diye 16'dan 8'e düşürdük
    per_device_eval_batch_size=8,        # Bunu da 8 yaptık
    num_train_epochs=3,                  
    weight_decay=0.01,
    eval_strategy="epoch",               
    save_strategy="epoch",               
    load_best_model_at_end=True,         
    metric_for_best_model="f1",          
    logging_steps=100,
    push_to_hub=False,
    fp16=False,  # <--- İŞTE KRİTİK NOKTA: Hatayı çözmek için False yaptık
)

In [ ]:
# 6. TRAINER OLUŞTURMA VE EĞİTİM
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer, 
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Eğitim NVIDIA T4 GPU üzerinde başlatılıyor...")
trainer.train()